# ReServe FP16 and AWQ compatibility check

This notebook performs **Step 1 only**: establish whether the pinned original
Qwen2.5-1.5B-Instruct checkpoint and its official AWQ checkpoint can supply
interchangeable decoder-layer variants. It creates a compatibility report and
a per-layer tensor manifest for the next switching-cost profiler.

**Kaggle setup:** select a GPU T4 session, enable Internet, and start with a
fresh kernel. If two T4s are visible, this notebook uses only GPU 0. Run the
cells in order. Cell 1 installs the recorded core stack only when necessary;
if it installs anything, restart the kernel once and run again. Downloads
reuse the Hugging Face cache. Earlier profiling notebooks are not needed.

Checks cover model and tokenizer compatibility, all saved tensors, AWQ packing,
changed FP16 companion tensors, tied weights, independently loaded endpoint
references, representative hybrids, exact restoration of original tensors,
and a small retained-KV high-to-low-to-high cycle. GPU checks are enabled by
default. The final cell creates a ZIP even when a preceding check fails.

The original checkpoint is stored as BF16. As in the completed FP16 profiles,
its floating tensors are converted to **FP16** for comparisons and execution.
Integer AWQ tensors are never cast to FP16.

The smoke test replaces complete decoder modules after synchronization.
Its temporary allocation layout is a compatibility reference. It does not
measure switch latency, prove an elastic page allocator, or choose a final
layer order. Successful checks do not establish the historical FP16 commit
used by the AWQ publisher or semantic output quality.


## 1. Check or install the recorded environment

If installation occurs, use Kaggle's **Restart session/kernel** action, then
run the notebook again. Keep the versions used by the successful profiles.


In [2]:
import importlib.metadata as md
import subprocess
import sys

INSTALL_IF_NEEDED = True
CORE_VERSIONS = {
    "torch": "2.6.0", "triton": "3.2.0",
    "transformers": "4.51.3", "autoawq": "0.2.9",
}

def installed_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

needs_core = any((installed_version(k) or "").split("+")[0] != v
                 for k, v in CORE_VERSIONS.items())
helper_pins = {
    "accelerate": "1.6.0", "safetensors": "0.5.3",
    "huggingface_hub": "0.30.2", "numpy": "1.26.4",
}
missing_helpers = [f"{k}=={v}" for k, v in helper_pins.items() if installed_version(k) is None]
print({k: installed_version(k) for k in CORE_VERSIONS})
if needs_core or missing_helpers:
    if not INSTALL_IF_NEEDED:
        raise RuntimeError("Enable INSTALL_IF_NEEDED in this cell, run it, and restart the kernel.")
    if needs_core:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
                               "--index-url", "https://download.pytorch.org/whl/cu124"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "transformers==4.51.3", "autoawq==0.2.9", "triton==3.2.0",
                               "accelerate==1.6.0", "safetensors==0.5.3",
                               "huggingface_hub==0.30.2", "tokenizers==0.21.1"])
    if missing_helpers:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_helpers])
    raise RuntimeError("Installation finished. Restart the kernel, then run the notebook from the top.")
print("Core versions match. Continue.")


{'torch': '2.6.0+cu124', 'triton': '3.2.0', 'transformers': '4.51.3', 'autoawq': '0.2.9'}
Core versions match. Continue.


## 2. Pin the two checkpoints and initialize the report

Leave these repository IDs and revisions unchanged. `RUN_GPU_CHECKS=False`
performs the static audit only; the final verdict will then be **INCOMPLETE**.
Logit tolerances are fixed here before seeing results. They compare two ways
of loading the **same endpoint**, not AWQ outputs against FP16 outputs.


In [3]:
import copy
import csv
import gc
import hashlib
import json
import math
import os
import platform
import re
import shutil
import struct
import traceback
from datetime import datetime, timezone
from pathlib import Path

import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer

SPECS = {
    "fp16": {"repo": "Qwen/Qwen2.5-1.5B-Instruct",
             "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"},
    "awq": {"repo": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
            "revision": "c5b72040769bfde6c5b562ae890f9657b0803f79"},
}
RUN_GPU_CHECKS = True
LOGIT_ATOL, LOGIT_RTOL = 0.02, 0.002
CHUNK_BYTES = 32 * 1024 * 1024
TEST_PROMPTS = [
    "Explain why a queue can grow when arrivals exceed service capacity.",
    "Compute 17 + 25. Give the answer and a short explanation.",
    "Write a Python function that returns the maximum item in a nonempty list.",
]
DECODE_STEPS = 2
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = working / f"compatibility_fp16_awq_{run_id}"
OUT.mkdir(parents=True, exist_ok=False)
STAGES = ["download", "config_tokenizer", "tensor_audit", "manifests",
          "gpu_endpoints", "gpu_hybrids", "gpu_continuity"]
REPORT = {
    "schema_version": 1, "run_id": run_id, "models": copy.deepcopy(SPECS),
    "checks": {k: {"status": "NOT_RUN"} for k in STAGES},
    "scope": "Checkpoint and decoder-layer compatibility; no cost or allocator validation",
    "provenance": {"awq_original_source_commit": "not_independently_established"},
    "logit_tolerances": {"atol": LOGIT_ATOL, "rtol": LOGIT_RTOL},
    "test_prompts": TEST_PROMPTS, "decode_steps": DECODE_STEPS,
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "cuda_runtime": torch.version.cuda,
                    "packages": {k: installed_version(k) for k in
                                 [*CORE_VERSIONS, *helper_pins, "tokenizers"]}},
}

def save_json(name, value):
    path = OUT / name
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    tmp.replace(path)

def save_csv(name, rows):
    if not rows:
        return
    with (OUT / name).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def record(stage, status, **details):
    REPORT["checks"][stage] = {"status": status, **details}
    save_json("compatibility_report.json", REPORT)
    print(f"{stage}: {status}")

def run_stage(stage, fn, dependencies=()):
    # A rerun must not leave downstream PASS results from an earlier attempt.
    for later in STAGES[STAGES.index(stage) + 1:]:
        REPORT["checks"][later] = {"status": "NOT_RUN", "reason": f"Rerun after {stage}"}
    missing = [d for d in dependencies if REPORT["checks"][d]["status"] != "PASS"]
    if missing:
        record(stage, "NOT_RUN", reason=f"Prerequisites did not pass: {missing}")
        return None
    record(stage, "RUNNING")
    try:
        result = fn()
        record(stage, "PASS", details=result if result is not None else {})
        return result
    except KeyboardInterrupt:
        record(stage, "INTERRUPTED", reason="Interrupted by the notebook user")
        raise
    except Exception as exc:
        record(stage, "FAIL", error_type=type(exc).__name__, error=str(exc))
        (OUT / f"{stage}_error.txt").write_text(traceback.format_exc())
        print(str(exc))
        print("The error was saved. Continue to the final export cell for the report ZIP.")
        return None

def require(condition, message):
    if not condition:
        raise AssertionError(message)

save_json("compatibility_report.json", REPORT)
print("Output folder:", OUT)


Output folder: /kaggle/working/compatibility_fp16_awq_20261127T191507_288845Z


## 3. Download the exact snapshots

This downloads approximately 4.3 GB of checkpoint data plus tokenizer files.
Only safetensors checkpoints and configuration/tokenizer files are used; no
remote model code is executed. Cached downloads are reused on repeat runs.


In [4]:
PATHS = {}

def download_snapshots():
    api = HfApi()
    for mode, spec in SPECS.items():
        info = api.model_info(spec["repo"], revision=spec["revision"])
        require(info.sha == spec["revision"], f"{mode}: resolved revision differs")
        names = {x.rfilename for x in info.siblings}
        if "model.safetensors.index.json" in names:
            index_path = hf_hub_download(spec["repo"], "model.safetensors.index.json",
                                         revision=spec["revision"])
            mapping = json.loads(Path(index_path).read_text())["weight_map"]
            weight_files = sorted(set(mapping.values()))
            require(all(x in names and x.endswith(".safetensors") for x in weight_files),
                    f"{mode}: index names missing or unsupported shards")
        else:
            require("model.safetensors" in names, f"{mode}: cannot locate a safetensors checkpoint")
            weight_files = ["model.safetensors"]
        small_files = [n for n in ["config.json", "tokenizer.json", "tokenizer_config.json",
                       "special_tokens_map.json", "vocab.json", "merges.txt", "added_tokens.json",
                       "generation_config.json", "model.safetensors.index.json"] if n in names]
        path = snapshot_download(spec["repo"], revision=spec["revision"],
                                 allow_patterns=small_files + weight_files, max_workers=2)
        PATHS[mode] = Path(path)
        REPORT["models"][mode].update({"resolved_revision": info.sha,
                                      "weight_files": weight_files})
        print(mode, "snapshot ready")
    save_json("resolved_models.json", REPORT["models"])
    return {k: str(v) for k, v in PATHS.items()}

run_stage("download", download_snapshots)


download: RUNNING


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

... (10 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

awq snapshot ready
download: PASS


{'fp16': '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
 'awq': '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct-AWQ/snapshots/c5b72040769bfde6c5b562ae890f9657b0803f79'}

## 4. Compare architecture and tokenizer behavior

Storage dtype and quantization metadata may differ. Architecture, attention,
rotary settings, vocabulary mappings, special IDs, and tokenizer processing
must agree. Original chat templates are compared and recorded separately.
Both modes then use the pinned FP16 tokenizer's chat template, so checkpoint
formatting differences cannot change their inputs. Core encoding mismatches
still block the GPU checks. Raw and effective tokenizer comparisons are both
saved in the report.


In [5]:
CONFIGS, TOKENIZERS = {}, {}

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()

def compare_tokenizer_records(records):
    a, b = records["fp16"], records["awq"]
    # A shared, explicitly selected template can resolve formatting differences.
    # It cannot resolve a different vocabulary, backend, or special-token map.
    template_fields = {"chat_template", "chat_ids"}
    core_differences = [k for k in sorted(set(a) | set(b))
                        if k not in template_fields and a.get(k) != b.get(k)]
    return {"tokenizer_core_match": not core_differences,
            "tokenizer_core_difference_fields": core_differences,
            "original_chat_templates_match": a.get("chat_template") == b.get("chat_template"),
            "original_chat_ids_match": a.get("chat_ids") == b.get("chat_ids")}

def config_tokenizer_check():
    for mode in SPECS:
        CONFIGS[mode] = json.loads((PATHS[mode] / "config.json").read_text())
        TOKENIZERS[mode] = AutoTokenizer.from_pretrained(
            str(PATHS[mode]), local_files_only=True, trust_remote_code=False, use_fast=True)
    ignored = {"torch_dtype", "transformers_version", "quantization_config",
               "_name_or_path", "_commit_hash"}
    differences = [{"field": k, "fp16": CONFIGS["fp16"].get(k), "awq": CONFIGS["awq"].get(k)}
                   for k in sorted(set(CONFIGS["fp16"]) | set(CONFIGS["awq"]))
                   if k not in ignored and CONFIGS["fp16"].get(k) != CONFIGS["awq"].get(k)]
    qcfg = CONFIGS["awq"].get("quantization_config", {})
    quant_ok = (qcfg.get("quant_method") == "awq" and qcfg.get("bits") == 4
                and qcfg.get("group_size") == 128 and qcfg.get("zero_point") is True
                and str(qcfg.get("version")).lower() == "gemm"
                and not qcfg.get("modules_to_not_convert"))
    tokenizer_records = {}
    for mode, tok in TOKENIZERS.items():
        fields = {k: getattr(tok, k) for k in ["bos_token_id", "eos_token_id", "pad_token_id",
                  "unk_token_id", "padding_side", "truncation_side", "model_max_length",
                  "clean_up_tokenization_spaces", "model_input_names"]}
        fields.update({"special_tokens": list(zip(tok.all_special_tokens, tok.all_special_ids)),
                       "chat_template": tok.chat_template,
                       "vocab_hash": canonical_hash(tok.get_vocab()),
                       "backend_hash": canonical_hash(json.loads(tok.backend_tokenizer.to_str()))})
        texts = TEST_PROMPTS + [" leading space\nnew line", "中文测试 café 123"]
        fields["encoded_tests"] = [tok.encode(t, add_special_tokens=True) for t in texts]
        fields["chat_ids"] = tok.apply_chat_template(
            [{"role": "user", "content": "Hello"}], tokenize=True, add_generation_prompt=True)
        tokenizer_records[mode] = fields
    comparison = compare_tokenizer_records(tokenizer_records)
    canonical_template = tokenizer_records["fp16"]["chat_template"]
    runtime_chat_ids = {}
    if comparison["tokenizer_core_match"]:
        for tok in TOKENIZERS.values():
            tok.chat_template = canonical_template
        tools = [{"type": "function", "function": {
            "name": "lookup", "description": "Look up a location",
            "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                           "required": ["query"]}}}]
        probes = [
            {"conversation": [{"role": "user", "content": "Hello"}]},
            {"conversation": [{"role": "system", "content": "Reply briefly."},
                          {"role": "user", "content": "Hello"}]},
            {"conversation": [{"role": "user", "content": "Look up Philadelphia."}], "tools": tools},
        ]
        for mode, tok in TOKENIZERS.items():
            runtime_chat_ids[mode] = [tok.apply_chat_template(
                **probe, tokenize=True, add_generation_prompt=True) for probe in probes]
    runtime_ok = (comparison["tokenizer_core_match"] and bool(canonical_template)
                  and runtime_chat_ids.get("fp16") == runtime_chat_ids.get("awq"))
    save_json("config_tokenizer.json", {"architecture_differences": differences,
              "allowed_storage_dtypes": {k: v.get("torch_dtype") for k, v in CONFIGS.items()},
              "quantization_config": qcfg, "quantization_supported": quant_ok,
              **comparison, "canonical_runtime_chat_ids_match": runtime_ok,
              "runtime_template_policy": {"source": SPECS["fp16"],
                  "sha256": canonical_hash(canonical_template),
                  "rule": "Use one pinned FP16 tokenizer and chat template for both model variants"},
              "runtime_chat_ids": runtime_chat_ids,
              "tokenizers": tokenizer_records})
    require(not differences, "Architecture/settings differ; see config_tokenizer.json")
    require(CONFIGS["fp16"].get("model_type") == "qwen2"
            and CONFIGS["fp16"].get("num_hidden_layers") == 28,
            "This notebook expects the pinned 28-layer Qwen2.5-1.5B architecture")
    require(quant_ok, "Unsupported AWQ packing configuration; see config_tokenizer.json")
    require(comparison["tokenizer_core_match"],
            f"Tokenizer encoding/settings differ: {comparison['tokenizer_core_difference_fields']}; see config_tokenizer.json")
    require(runtime_ok, "Canonical tokenizer/template inputs differ; see config_tokenizer.json")
    if not comparison["original_chat_templates_match"]:
        print("Original checkpoint chat templates differ; recorded as a formatting difference.")
        print("Both model variants will use the pinned FP16 tokenizer and chat template.")
    return {"num_layers": 28, "tokenizer_core": "exact match on recorded properties",
            "chat_template_policy": "pinned FP16 template shared by both modes",
            "original_chat_templates_match": comparison["original_chat_templates_match"],
            "fp16_comparison": "all floating tensors converted to torch.float16"}

run_stage("config_tokenizer", config_tokenizer_check, dependencies=("download",))


config_tokenizer: RUNNING
Original checkpoint chat templates differ; recorded as a formatting difference.
Both model variants will use the pinned FP16 tokenizer and chat template.
config_tokenizer: PASS


{'num_layers': 28,
 'tokenizer_core': 'exact match on recorded properties',
 'chat_template_policy': 'pinned FP16 template shared by both modes',
 'original_chat_templates_match': False,
 'fp16_comparison': 'all floating tensors converted to torch.float16'}

## 5. Audit every stored tensor without loading both models into GPU memory

The audit streams CPU tensors in small chunks. It records shapes, dtypes,
raw hashes, hashes after FP16 conversion, and nonfinite values. Tied embedding
and output weights are resolved explicitly when one safetensors alias is
omitted. This cell may take several minutes; progress is printed.


In [6]:
class TensorStore:
    def __init__(self, path):
        self.path = Path(path)
        self.headers = {}
        self.metadata = {}
        index_path = self.path / "model.safetensors.index.json"
        mapping = json.loads(index_path.read_text())["weight_map"] if index_path.exists() else None
        files = sorted(set(mapping.values())) if mapping else ["model.safetensors"]
        for name in files:
            fullpath = self.path / name
            with fullpath.open("rb") as f:
                prefix = f.read(8)
                require(len(prefix) == 8, f"Truncated safetensors file: {name}")
                length = struct.unpack("<Q", prefix)[0]
                require(0 < length <= min(100_000_000, fullpath.stat().st_size - 8),
                        f"Invalid safetensors header length: {name}")
                header = json.loads(f.read(length))
            self.metadata[name] = header.pop("__metadata__", {})
            intervals = []
            for key, entry in header.items():
                require(key not in self.headers, f"Duplicate checkpoint tensor: {key}")
                lo, hi = entry["data_offsets"]
                require(0 <= lo <= hi <= fullpath.stat().st_size - 8 - length,
                        f"Invalid offsets: {key}")
                intervals.append((lo, hi, key))
                self.headers[key] = {**entry, "file": name, "stored_bytes": hi - lo}
            ordered = sorted(intervals)
            require(all(a[1] <= b[0] for a, b in zip(ordered, ordered[1:])),
                    f"Overlapping serialized tensor storage in {name}")
        if mapping:
            require(set(mapping) == set(self.headers), "Index/header tensor names differ")
            require(all(self.headers[k]["file"] == v for k, v in mapping.items()),
                    "Index/header shard assignment differs")

    def load(self, key):
        with safe_open(str(self.path / self.headers[key]["file"]), framework="pt", device="cpu") as f:
            return f.get_tensor(key)

    def chunks(self, key):
        entry = self.headers[key]
        with safe_open(str(self.path / entry["file"]), framework="pt", device="cpu") as f:
            shape = entry["shape"]
            if not shape or math.prod(shape) == 0:
                yield f.get_tensor(key)
                return
            per_row = max(1, entry["stored_bytes"] // max(1, shape[0]))
            rows = max(1, CHUNK_BYTES // per_row)
            view = f.get_slice(key)
            for start in range(0, shape[0], rows):
                yield view[start:min(shape[0], start + rows)]

def tensor_digest(tensor):
    value = tensor.detach().cpu().contiguous().reshape(-1)
    return hashlib.sha256(value.view(torch.uint8).numpy().tobytes()).hexdigest()

def normalized_tensor(tensor):
    return tensor.to(torch.float16) if tensor.is_floating_point() else tensor

def audit_tensor(store, key):
    raw_hash, normalized_hash = hashlib.sha256(), hashlib.sha256()
    finite, positive = True, True
    normalized_bytes = 0
    for part in store.chunks(key):
        raw_hash.update(part.contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
        value = normalized_tensor(part).contiguous()
        normalized_hash.update(value.reshape(-1).view(torch.uint8).numpy().tobytes())
        normalized_bytes += value.numel() * value.element_size()
        if value.is_floating_point():
            finite = finite and bool(torch.isfinite(value).all())
            if key.endswith(".scales"):
                positive = positive and bool((value > 0).all())
    return {"raw_sha256": raw_hash.hexdigest(), "normalized_sha256": normalized_hash.hexdigest(),
            "finite_after_conversion": finite, "positive_scales": positive,
            "normalized_bytes": normalized_bytes}

def layer_number(key):
    match = re.match(r"^model\.layers\.(\d+)\.", key)
    return int(match.group(1)) if match else None

STORES, INVENTORIES, AUDITS = {}, {}, {}

def tensor_audit():
    nonfinite = []
    for mode in SPECS:
        store = STORES[mode] = TensorStore(PATHS[mode])
        INVENTORIES[mode] = []
        AUDITS[mode] = {}
        for i, key in enumerate(sorted(store.headers), 1):
            entry = store.headers[key]
            audit = audit_tensor(store, key)
            AUDITS[mode][key] = audit
            row = {"mode": mode, "name": key, "layer": layer_number(key),
                   "shape": json.dumps(entry["shape"]), "stored_dtype": entry["dtype"],
                   "shard": entry["file"], "stored_bytes": entry["stored_bytes"], **audit}
            INVENTORIES[mode].append(row)
            if not audit["finite_after_conversion"] or not audit["positive_scales"]:
                nonfinite.append({"mode": mode, "name": key})
            if i % 100 == 0 or i == len(store.headers):
                print(f"{mode}: {i}/{len(store.headers)} tensors checked", flush=True)
        save_csv(f"inventory_{mode}.csv", INVENTORIES[mode])
        save_json(f"safetensors_metadata_{mode}.json", store.metadata)
    require(not nonfinite, f"Nonfinite tensor or nonpositive AWQ scales: {nonfinite[:10]}")
    return {mode: {"stored_tensors": len(rows),
                   "normalized_stored_bytes": sum(r["normalized_bytes"] for r in rows)}
            for mode, rows in INVENTORIES.items()}

run_stage("tensor_audit", tensor_audit, dependencies=("config_tokenizer",))


tensor_audit: RUNNING
fp16: 100/338 tensors checked
fp16: 200/338 tensors checked
fp16: 300/338 tensors checked
fp16: 338/338 tensors checked
awq: 100/731 tensors checked
awq: 200/731 tensors checked
awq: 300/731 tensors checked
awq: 400/731 tensors checked
awq: 500/731 tensors checked
awq: 600/731 tensors checked
awq: 700/731 tensors checked
awq: 731/731 tensors checked
tensor_audit: PASS


{'fp16': {'stored_tensors': 338, 'normalized_stored_bytes': 3087428608},
 'awq': {'stored_tensors': 731, 'normalized_stored_bytes': 1614472192}}

## 6. Validate the layer boundary and write variant manifests

Differences inside a decoder layer are retained with that variant, including
any changed biases or normalization tensors. Shared tensors outside decoder
layers must match exactly after FP16 conversion. Unknown or incompatible
state blocks the gate. A saved embedding alias is not counted twice.

The manifest contains both the full-layer reference payload and the smaller
candidate payload consisting of quantized linears plus changed companions.
These byte counts are tensor inventories, not measured aligned GPU savings.


In [8]:
LINEARS = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
           "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"]
GLOBAL_KEYS = {"model.embed_tokens.weight", "lm_head.weight", "model.norm.weight"}
MANIFESTS = []

def effective_global_key(mode, key):
    headers = STORES[mode].headers
    if key in headers:
        return key
    if CONFIGS[mode].get("tie_word_embeddings"):
        alias = {"lm_head.weight": "model.embed_tokens.weight",
                 "model.embed_tokens.weight": "lm_head.weight"}.get(key)
        if alias in headers:
            return alias
    return None

def same_tensor(mode_a, key_a, mode_b, key_b):
    a, b = STORES[mode_a].headers[key_a], STORES[mode_b].headers[key_b]
    return (a["shape"] == b["shape"]
            and AUDITS[mode_a][key_a]["normalized_sha256"] == AUDITS[mode_b][key_b]["normalized_sha256"])

def manifest_tensor(mode, key):
    h, a = STORES[mode].headers[key], AUDITS[mode][key]
    return {"name": key, "shape": h["shape"], "stored_dtype": h["dtype"],
            "execution_dtype": "float16" if h["dtype"] in {"BF16", "F16", "F32", "F64"} else h["dtype"],
            "bytes": a["normalized_bytes"], "normalized_sha256": a["normalized_sha256"]}

def make_manifests():
    MANIFESTS.clear()
    problems, globals_report, companions, layer_rows = [], [], [], []
    for mode, store in STORES.items():
        unknown = [k for k in store.headers if layer_number(k) is None and k not in GLOBAL_KEYS]
        if unknown:
            problems.append(f"{mode}: unhandled global tensors {unknown}")
        layers = {layer_number(k) for k in store.headers if layer_number(k) is not None}
        if layers != set(range(28)):
            problems.append(f"{mode}: unexpected layer IDs")
        if CONFIGS[mode].get("tie_word_embeddings"):
            a = effective_global_key(mode, "model.embed_tokens.weight")
            b = effective_global_key(mode, "lm_head.weight")
            if a is None or b is None or not same_tensor(mode, a, mode, b):
                problems.append(f"{mode}: tied embedding/output values are inconsistent")
    for key in sorted(GLOBAL_KEYS):
        a, b = effective_global_key("fp16", key), effective_global_key("awq", key)
        equal = a is not None and b is not None and same_tensor("fp16", a, "awq", b)
        globals_report.append({"logical_name": key, "fp16_source": a, "awq_source": b,
                               "identical_after_fp16_conversion": equal})
        if not equal:
            problems.append(f"Global shared tensor differs or is missing: {key}")
    for j in range(28):
        prefix = f"model.layers.{j}."
        keys = {mode: {k for k in store.headers if k.startswith(prefix)} for mode, store in STORES.items()}
        dense_weights = {prefix + p + ".weight" for p in LINEARS}
        packed = {prefix + p + "." + s for p in LINEARS for s in ["qweight", "qzeros", "scales"]}
        expected_companions = {prefix + p + ".bias" for p in LINEARS[:3]}
        expected_companions |= {prefix + "input_layernorm.weight", prefix + "post_attention_layernorm.weight"}
        if keys["fp16"] != dense_weights | expected_companions:
            problems.append(f"Layer {j}: unexpected FP16 state keys")
        if keys["awq"] != packed | expected_companions:
            problems.append(f"Layer {j}: unexpected AWQ state keys")
        for p in LINEARS:
            dense_key = prefix + p + ".weight"
            qkeys = {s: prefix + p + "." + s for s in ["qweight", "qzeros", "scales"]}
            if dense_key not in keys["fp16"] or not set(qkeys.values()) <= keys["awq"]:
                continue
            shape = STORES["fp16"].headers[dense_key]["shape"]
            if len(shape) != 2:
                problems.append(f"Not a matrix: {dense_key}")
                continue
            nout, nin = shape
            expected = {"qweight": ([nin, nout // 8], "I32"),
                        "qzeros": ([nin // 128, nout // 8], "I32"),
                        "scales": ([nin // 128, nout], "F16")}
            if nin % 128 or nout % 8:
                problems.append(f"Packing alignment fails: {dense_key}")
            for s, (expected_shape, dtype) in expected.items():
                h = STORES["awq"].headers[qkeys[s]]
                if h["shape"] != expected_shape or h["dtype"] != dtype:
                    problems.append(f"Packed shape/dtype mismatch: {qkeys[s]}")
        changed = []
        for key in sorted(expected_companions & keys["fp16"] & keys["awq"]):
            a, b = STORES["fp16"].headers[key], STORES["awq"].headers[key]
            if a["shape"] != b["shape"] or a["dtype"] not in {"BF16", "F16", "F32"} or b["dtype"] != "F16":
                problems.append(f"Companion layout/dtype mismatch: {key}")
            equal = same_tensor("fp16", key, "awq", key)
            companions.append({"layer": j, "name": key, "equal_after_fp16_conversion": equal,
                               "must_replace_with_variant": not equal})
            if not equal:
                changed.append(key)
        variants = {}
        for mode, payload in [("fp16", dense_weights), ("awq", packed)]:
            required = [manifest_tensor(mode, k) for k in sorted(keys[mode])]
            movable = [manifest_tensor(mode, k) for k in sorted((payload | set(changed)) & keys[mode])]
            variants[mode] = {"all_layer_tensors": required, "candidate_variant_tensors": movable,
                              "full_layer_bytes": sum(x["bytes"] for x in required),
                              "candidate_payload_bytes": sum(x["bytes"] for x in movable),
                              "linear_dispatch": "torch.nn.Linear" if mode == "fp16" else "AutoAWQ WQLinear_GEMM/Triton"}
        MANIFESTS.append({"layer": j, "boundary": prefix.rstrip("."), "changed_companions": changed,
                          "variants": variants})
        layer_rows.append({"layer": j, "fp16_layer_bytes": variants["fp16"]["full_layer_bytes"],
                           "awq_layer_bytes": variants["awq"]["full_layer_bytes"],
                           "raw_layer_difference_bytes": variants["fp16"]["full_layer_bytes"] - variants["awq"]["full_layer_bytes"],
                           "changed_companions": len(changed)})
    save_json("global_tensor_compatibility.json", globals_report)
    save_json("variant_manifests.json", {"layers": MANIFESTS,
              "basis": "FP16-converted floating values; all stored integer values retained",
              "allocator_validation": "not_performed", "problems": problems})
    save_csv("companion_tensor_comparison.csv", companions)
    save_csv("layer_inventory.csv", layer_rows)
    require(not problems, "; ".join(problems[:12]) + (" ..." if len(problems) > 12 else ""))
    return {"layers_checked": len(MANIFESTS), "changed_companion_tensors": sum(not c["equal_after_fp16_conversion"] for c in companions),
            "shared_global_tensors": "match", "replacement_boundary": "one complete decoder layer"}

run_stage("manifests", make_manifests, dependencies=("tensor_audit",))


manifests: RUNNING
manifests: PASS


{'layers_checked': 28,
 'changed_companion_tensors': 56,
 'shared_global_tensors': 'match',
 'replacement_boundary': 'one complete decoder layer'}

## 7. Define small GPU compatibility checks

This uses the same unfused GEMM/Triton and SDPA paths as the recorded profiles.
Independent FP16 and AWQ endpoint references are loaded **one at a time**.
The hybrid model is subsequently built from the original FP16 checkpoint.
Fixed continuation token IDs keep endpoint comparisons on identical inputs.

These are short correctness checks, not timing measurements or a quality
benchmark. Whole-layer reference replacement can temporarily retain both
old and new layer storage. No reclaimed-page behavior is inferred from it.


In [13]:
DEVICE = torch.device("cuda:0")
REFERENCES, GPU_ROWS = {}, []
hybrid_model = None

def prepare_gpu():
    require(torch.cuda.is_available(), "No CUDA GPU found. Enable a T4 or set RUN_GPU_CHECKS=False.")
    torch.cuda.set_device(0)
    require("T4" in torch.cuda.get_device_name(0), "Use a Tesla T4 to match the recorded calibration stack.")
    require(torch.version.cuda == "12.4", "CUDA runtime differs from the recorded cu124 stack.")
    for name, version in CORE_VERSIONS.items():
        require((installed_version(name) or "").split("+")[0] == version,
                f"Version mismatch: {name}; rerun Cell 1 and restart")
    torch.manual_seed(20261127)
    torch.cuda.manual_seed_all(20261127)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    import awq.modules.linear.gemm as gemm
    require(gemm.TRITON_AVAILABLE, "AutoAWQ Triton backend is unavailable")
    # AutoAWQ prefers its optional compiled extension when installed. Force the
    # recorded Triton path so a preinstalled extension cannot change the check.
    extension_was_present = gemm.awq_ext is not None
    gemm.awq_ext = None
    props = torch.cuda.get_device_properties(0)
    REPORT["environment"].update({"gpu": props.name, "gpu_total_bytes": props.total_memory,
                                   "visible_gpus": torch.cuda.device_count(), "selected_gpu": 0,
                                   "attention": "sdpa", "awq_dispatch": "unfused GEMM/Triton",
                                   "optional_awq_extension_was_present": extension_was_present})
    save_json("compatibility_report.json", REPORT)

def load_endpoint(mode):
    from transformers import AutoModelForCausalLM, AwqConfig
    options = dict(torch_dtype=torch.float16, attn_implementation="sdpa",
                   local_files_only=True, trust_remote_code=False, use_safetensors=True,
                   low_cpu_mem_usage=True, output_loading_info=True)
    if mode == "awq":
        options["quantization_config"] = AwqConfig(
            bits=4, group_size=128, zero_point=True, version="gemm", backend="autoawq", do_fuse=False)
        options["device_map"] = {"": 0}
    model, info = AutoModelForCausalLM.from_pretrained(str(PATHS[mode]), **options)
    # A tied alias may be absent from the serialized checkpoint. No other
    # missing, unexpected, or mismatched state is accepted.
    permitted_missing = {"lm_head.weight", "model.embed_tokens.weight"} if CONFIGS[mode].get("tie_word_embeddings") else set()
    require(not (set(info.get("missing_keys", [])) - permitted_missing), f"{mode}: missing model state {info}")
    require(not info.get("unexpected_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"),
            f"{mode}: checkpoint loading mismatch {info}")
    if mode == "fp16":
        model.to(DEVICE)
    from accelerate.hooks import remove_hook_from_module
    remove_hook_from_module(model, recurse=True)
    model.eval().requires_grad_(False)
    require(model.config._attn_implementation == "sdpa", "Unexpected attention implementation")
    return model

def ownership_report(model):
    aliases, storage_owners = [], {}
    tensors = list(model.named_parameters(remove_duplicate=False)) + list(model.named_buffers(remove_duplicate=False))
    for name, t in tensors:
        require(not t.is_meta and t.device == DEVICE, f"Wrong device or meta tensor: {name}")
        if layer_number(name) is not None and t.is_floating_point():
            require(t.dtype == torch.float16, f"Layer floating tensor is not FP16: {name}")
        if t.numel():
            identity = (str(t.device), t.untyped_storage().data_ptr(), t.untyped_storage().nbytes())
            storage_owners.setdefault(identity, []).append(name)
    for names in storage_owners.values():
        if len(names) <= 1:
            continue
        scopes = {layer_number(n) for n in names}
        require(len(scopes) == 1, f"Tensor storage crosses decoder boundaries: {names}")
        if scopes == {None}:
            require(set(names) <= {"model.embed_tokens.weight", "lm_head.weight"},
                    f"Unhandled global storage alias: {names}")
        aliases.append(names)
    if CONFIGS["fp16"].get("tie_word_embeddings"):
        require(model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr(),
                "Input and output embeddings are not actually tied")
    return {"storage_aliases": aliases, "all_state_on_gpu_0": True}

def cache_pairs(cache):
    require(hasattr(cache, "to_legacy_cache"), "Expected a Transformers DynamicCache")
    pairs = cache.to_legacy_cache()
    require(len(pairs) == 28, "Unexpected number of KV layers")
    return pairs

def check_cache(cache, expected_length):
    require(cache.get_seq_length() == expected_length, "Cache token count is incorrect")
    cfg = CONFIGS["fp16"]
    expected_shape = (1, cfg["num_key_value_heads"], expected_length,
                      cfg["hidden_size"] // cfg["num_attention_heads"])
    for j, pair in enumerate(cache_pairs(cache)):
        for t in pair:
            require(tuple(t.shape) == expected_shape and t.dtype == torch.float16 and t.device == DEVICE,
                    f"Layer {j}: KV shape, dtype, or device mismatch")
            require(bool(torch.isfinite(t).all()), f"Layer {j}: nonfinite KV values")

def cache_snapshot(cache):
    # Only short test contexts are copied. These CPU clones are never reused as
    # model inputs and cannot be mutated by an in-place cache update.
    return [(k.detach().cpu().clone(), v.detach().cpu().clone()) for k, v in cache_pairs(cache)]

def check_cache_prefix(cache, snapshot):
    for j, ((k, v), (old_k, old_v)) in enumerate(zip(cache_pairs(cache), snapshot)):
        length = old_k.shape[-2]
        require(torch.equal(k[..., :length, :].cpu(), old_k)
                and torch.equal(v[..., :length, :].cpu(), old_v),
                f"Layer {j}: previously cached values were changed")

@torch.inference_mode()
def decode_one(model, cache, token_id):
    length = cache.get_seq_length()
    snapshot = cache_snapshot(cache)
    result = model(input_ids=torch.tensor([[token_id]], device=DEVICE),
                   attention_mask=torch.ones((1, length + 1), dtype=torch.long, device=DEVICE),
                   position_ids=torch.tensor([[length]], device=DEVICE),
                   cache_position=torch.tensor([length], device=DEVICE),
                   past_key_values=cache, use_cache=True, return_dict=True, logits_to_keep=1)
    require(result.past_key_values is cache, "The cache object was replaced")
    check_cache(cache, length + 1)
    check_cache_prefix(cache, snapshot)
    require(bool(torch.isfinite(result.logits).all()), "Nonfinite decode logits")
    return result.logits[:, -1].detach().float().cpu()

@torch.inference_mode()
def fixed_trace(model):
    tok = TOKENIZERS["fp16"]
    continuation = tok.encode(" A short answer.", add_special_tokens=False)[:DECODE_STEPS]
    require(len(continuation) == DECODE_STEPS, "Not enough fixed continuation tokens")
    collected = []
    for prompt in TEST_PROMPTS:
        ids = tok(prompt, return_tensors="pt", add_special_tokens=True)["input_ids"].to(DEVICE)
        require(1 < ids.shape[1] < 128, "Smoke-test prompt outside the intended short range")
        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=True,
                       return_dict=True, logits_to_keep=1)
        require(bool(torch.isfinite(output.logits).all()), "Nonfinite prefill logits")
        cache = output.past_key_values
        check_cache(cache, ids.shape[1])
        collected.append(output.logits[:, -1].detach().float().cpu())
        for token_id in continuation:
            collected.append(decode_one(model, cache, token_id))
        del output, cache, ids
    torch.cuda.synchronize(DEVICE)
    return torch.cat(collected, dim=0)

def compare_logits(name, actual, expected):
    require(actual.shape == expected.shape, f"{name}: output shape mismatch")
    delta = (actual - expected).abs()
    passed = bool(torch.allclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL))
    row = {"test": name, "passed": passed, "max_absolute_logit_difference": float(delta.max()),
           "mean_absolute_logit_difference": float(delta.mean()), "atol": LOGIT_ATOL, "rtol": LOGIT_RTOL}
    GPU_ROWS.append(row)
    save_csv("gpu_endpoint_comparisons.csv", GPU_ROWS)
    require(passed, f"{name}: endpoint mismatch; see gpu_endpoint_comparisons.csv")
    return row

def independent_endpoints():
    prepare_gpu()
    for mode in ["fp16", "awq"]:
        model = None
        try:
            model = load_endpoint(mode)
            ownership = ownership_report(model)
            REFERENCES[mode] = fixed_trace(model)
            save_json(f"runtime_ownership_{mode}.json", ownership)
            print(mode, "reference: finite prefill, decode, and FP16 KV")
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    return {"endpoints": list(REFERENCES), "one_gpu_endpoint_at_a_time": True}

if RUN_GPU_CHECKS:
    run_stage("gpu_endpoints", independent_endpoints, dependencies=("manifests",))
else:
    record("gpu_endpoints", "NOT_RUN", reason="RUN_GPU_CHECKS=False; static audit only")


gpu_endpoints: RUNNING
fp16 reference: finite prefill, decode, and FP16 KV


/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['version', 'fuse_max_seq_len', 'exllama_config', 'modules_to_fuse', 'do_fuse']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)


awq reference: finite prefill, decode, and FP16 KV
gpu_endpoints: PASS


## 8. Reconstruct decoder variants and test representative hybrids

Each replacement loads **all** of that decoder layer's required state with
`strict=True`, including its own normalization and bias values. The tested
mixed-layer sets are fixtures; they are not the final LIS quantization order.
Full-AWQ and full-FP16 endpoints must match independently loaded references.
After restoration, every original decoder tensor is checked against its
saved checkpoint value after FP16 conversion.


In [14]:
def build_layer(mode, j, device):
    from transformers.models.qwen2.modeling_qwen2 import Qwen2DecoderLayer
    from awq.modules.linear.gemm import WQLinear_GEMM
    config = AutoConfig.from_pretrained(str(PATHS["fp16"]), local_files_only=True, trust_remote_code=False)
    config._attn_implementation = "sdpa"
    with torch.device("meta"):
        layer = Qwen2DecoderLayer(config, layer_idx=j)
        if mode == "awq":
            for path in LINEARS:
                parent_path, attr = path.rsplit(".", 1)
                parent = layer.get_submodule(parent_path)
                dense = getattr(parent, attr)
                packed = WQLinear_GEMM(w_bit=4, group_size=128,
                                      in_features=dense.in_features, out_features=dense.out_features,
                                      bias=dense.bias is not None, dev="meta")
                setattr(parent, attr, packed)
    prefix = f"model.layers.{j}."
    state = {k[len(prefix):]: normalized_tensor(STORES[mode].load(k)).clone()
             for k in STORES[mode].headers if k.startswith(prefix)}
    layer.load_state_dict(state, strict=True, assign=True)
    layer.eval().requires_grad_(False)
    require(not any(t.is_meta for t in list(layer.parameters()) + list(layer.buffers())),
            f"Layer {j}: incomplete tensor materialization")
    return layer.to(device)

def verify_layer_values(model, mode, j):
    prefix = f"model.layers.{j}."
    state = model.model.layers[j].state_dict()
    expected = {k[len(prefix):] for k in STORES[mode].headers if k.startswith(prefix)}
    require(set(state) == expected, f"Layer {j}: execution/checkpoint key mismatch")
    for key, value in state.items():
        require(tensor_digest(value) == AUDITS[mode][prefix + key]["normalized_sha256"],
                f"Layer {j}: loaded/restored value differs: {key}")

def replace_layer(model, mode, j):
    # Deliberately synchronous compatibility reference. No time or memory
    # observations from this function are treated as switching-cost profiles.
    torch.cuda.synchronize(DEVICE)
    replacement = build_layer(mode, j, DEVICE)
    torch.cuda.synchronize(DEVICE)
    old = model.model.layers[j]
    model.model.layers[j] = replacement
    del old
    verify_layer_values(model, mode, j)

def hybrid_checks():
    global hybrid_model
    hybrid_model = load_endpoint("fp16")
    compare_logits("hybrid_q0_vs_independent_fp16", fixed_trace(hybrid_model), REFERENCES["fp16"])
    current = set()
    tests = [({0}, "fixture_layer_0"), ({0, 13, 27}, "fixture_layers_0_13_27"),
             (set(range(28)), "all_layers_awq")]
    for target, name in tests:
        for j in sorted(current - target, reverse=True):
            replace_layer(hybrid_model, "fp16", j)
        for j in sorted(target - current):
            replace_layer(hybrid_model, "awq", j)
        current = set(target)
        logits = fixed_trace(hybrid_model)
        ownership = ownership_report(hybrid_model)
        if len(target) == 28:
            compare_logits("hybrid_all_awq_vs_independent_awq", logits, REFERENCES["awq"])
        save_json(f"hybrid_{name}.json", {"awq_layers": sorted(target), "finite_prefill_decode": True,
                                         "fp16_kv_checked": True, **ownership})
        print(name, "passed")
    for j in sorted(current, reverse=True):
        replace_layer(hybrid_model, "fp16", j)
    for j in range(28):
        verify_layer_values(hybrid_model, "fp16", j)
    compare_logits("restored_fp16_fresh_cache", fixed_trace(hybrid_model), REFERENCES["fp16"])
    return {"representative_awq_sets": [[0], [0, 13, 27], list(range(28))],
            "all_original_decoder_tensors_restored_exactly": True,
            "ordering": "compatibility fixtures, not final LIS path"}

if RUN_GPU_CHECKS:
    run_stage("gpu_hybrids", hybrid_checks, dependencies=("gpu_endpoints",))
else:
    record("gpu_hybrids", "NOT_RUN", reason="RUN_GPU_CHECKS=False")


gpu_hybrids: RUNNING
fixture_layer_0 passed
fixture_layers_0_13_27 passed
gpu_hybrids: FAIL
hybrid_all_awq_vs_independent_awq: endpoint mismatch; see gpu_endpoint_comparisons.csv
The error was saved. Continue to the final export cell for the report ZIP.


## 9. Verify retained KV through a small high to low to high cycle

Start a request in FP16, replace three layers with their AWQ variants, decode
one more token, restore the original FP16 layers, and decode again. Cached
prefixes, positions, token counts, and dtype must remain intact. A final
fresh-cache FP16 check is separate because mixed-history KV need not match an
uninterrupted FP16 response.


In [11]:
@torch.inference_mode()
def continuity_check():
    require(hybrid_model is not None, "Hybrid model is unavailable")
    tok = TOKENIZERS["fp16"]
    ids = tok(TEST_PROMPTS[0], return_tensors="pt")["input_ids"].to(DEVICE)
    first = hybrid_model(input_ids=ids, attention_mask=torch.ones_like(ids),
                         use_cache=True, return_dict=True, logits_to_keep=1)
    require(bool(torch.isfinite(first.logits).all()), "Nonfinite initial logits")
    cache = first.past_key_values
    length = ids.shape[1]
    check_cache(cache, length)
    snapshot = cache_snapshot(cache)
    changed_layers = [0, 13, 27]
    for j in changed_layers:
        replace_layer(hybrid_model, "awq", j)
    check_cache(cache, length)
    check_cache_prefix(cache, snapshot)
    continuation = tok.encode(" A short answer.", add_special_tokens=False)
    decode_one(hybrid_model, cache, continuation[0])
    snapshot = cache_snapshot(cache)
    for j in reversed(changed_layers):
        replace_layer(hybrid_model, "fp16", j)
    check_cache(cache, length + 1)
    check_cache_prefix(cache, snapshot)
    decode_one(hybrid_model, cache, continuation[1])
    check_cache(cache, length + 2)
    del first, cache, snapshot, ids
    compare_logits("post_cycle_fp16_fresh_cache", fixed_trace(hybrid_model), REFERENCES["fp16"])
    return {"awq_layers": changed_layers, "prompt_tokens": length,
            "tokens_appended": 2, "fp16_kv_prefixes_preserved": True,
            "original_tensor_values_verified_on_restore": True,
            "page_allocator_or_async_switch_validation": False}

if RUN_GPU_CHECKS:
    run_stage("gpu_continuity", continuity_check, dependencies=("gpu_hybrids",))
else:
    record("gpu_continuity", "NOT_RUN", reason="RUN_GPU_CHECKS=False")

if hybrid_model is not None:
    del hybrid_model
    hybrid_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


gpu_continuity: NOT_RUN


## 9b. Diagnose a full AWQ endpoint mismatch if Step 8 failed

This cell runs only when the endpoint references passed and Step 8 failed.
For an existing failed session, copy this entire code cell into Kaggle, run
it, then rerun Step 10 to export the ZIP. Steps 1 through 6 need not be repeated
while their variables remain in the current kernel.

It compares repeatability within each model, all model tensors and runtime
state between models, and endpoint outputs with a serial split-K reduction.
The serial reduction is a correctness diagnostic only: the original
kernel is restored afterward. It changes neither the profiling path nor the
original tolerances or failed verdict. A diagnostic match alone does not
automatically pass the main compatibility gate.

In [15]:
from contextlib import contextmanager

def diagnostic_logit_metrics(name, actual, expected):
    require(actual.shape == expected.shape, f"Diagnostic output shape differs: {name}")
    require(bool(torch.isfinite(actual).all()) and bool(torch.isfinite(expected).all()),
            f"Diagnostic output is nonfinite: {name}")
    delta = (actual - expected).abs()
    limit = LOGIT_ATOL + LOGIT_RTOL * expected.abs()
    return {"test": name, "passed_original_tolerance": bool((delta <= limit).all()),
            "max_absolute_difference": float(delta.max()), "mean_absolute_difference": float(delta.mean()),
            "elements_outside_tolerance": int((delta > limit).sum()),
            "elements_compared": actual.numel(), "atol": LOGIT_ATOL, "rtol": LOGIT_RTOL}

@contextmanager
def diagnostic_serial_awq_reduction():
    import awq.modules.linear.gemm as gemm
    require(gemm.TRITON_AVAILABLE, "The serial diagnostic requires the recorded Triton backend")
    original = gemm.awq_gemm_triton
    def serial(input, qweight, scales, qzeros, split_k_iters=8, **kwargs):
        return original(input, qweight, scales, qzeros, split_k_iters=1, **kwargs)
    torch.cuda.synchronize(DEVICE)
    gemm.awq_gemm_triton = serial
    try:
        yield
    finally:
        # Restore the profiling path even if a diagnostic raises or is interrupted.
        try:
            torch.cuda.synchronize(DEVICE)
        finally:
            gemm.awq_gemm_triton = original

def diagnostic_runtime_snapshot(model):
    tensor_records = {}
    def add(name, value, role):
        tensor_records[name] = {"role": role, "shape": list(value.shape), "dtype": str(value.dtype),
                                "stride": list(value.stride()), "sha256": tensor_digest(value)}
    for name, value in model.state_dict().items():
        add(name, value, "checkpoint_state")
    for name, value in model.named_buffers(remove_duplicate=False):
        if name not in tensor_records:
            add(name, value, "nonpersistent_buffer")
    modules = {}
    for name, module in model.named_modules():
        values = {"class": type(module).__module__ + "." + type(module).__qualname__,
                  "training": module.training}
        for attr in ["in_features", "out_features", "w_bit", "group_size", "layer_idx", "head_dim",
                     "scaling", "variance_epsilon", "rope_type", "attention_scaling",
                     "max_seq_len_cached", "original_max_seq_len"]:
            value = getattr(module, attr, None)
            if isinstance(value, (int, float, str, bool)):
                values[attr] = value
        original_freq = getattr(module, "original_inv_freq", None)
        if isinstance(original_freq, torch.Tensor):
            add(name + ".original_inv_freq", original_freq, "unregistered_rotary_attribute")
        modules[name] = values
    config_fields = ["model_type", "hidden_size", "intermediate_size", "num_hidden_layers",
                     "num_attention_heads", "num_key_value_heads", "head_dim", "rms_norm_eps",
                     "rope_theta", "rope_scaling", "max_position_embeddings", "use_sliding_window",
                     "sliding_window", "max_window_layers", "tie_word_embeddings", "_attn_implementation"]
    config = {k: getattr(model.config, k, None) for k in config_fields}
    return {"tensors": tensor_records, "modules": modules, "execution_config": config}

def diagnostic_state_differences(reference, reconstructed):
    differences = []
    for category in ["tensors", "modules", "execution_config"]:
        a, b = reference[category], reconstructed[category]
        for name in sorted(set(a) | set(b)):
            if a.get(name) != b.get(name):
                differences.append({"category": category, "name": name,
                                    "independent_awq": a.get(name), "reconstructed_awq": b.get(name)})
    return differences

def diagnose_awq_hybrid_failure():
    global hybrid_model
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    relative = f"hybrid_diagnostic/{stamp}"
    (OUT / relative).mkdir(parents=True, exist_ok=False)
    summary = {"status": "RUNNING", "folder": relative, "native_split_k": 8, "diagnostic_split_k": 1,
               "native_repeats": 3, "serial_repeats": 2, "warmups_per_path": 1,
               "original_tolerance": {"atol": LOGIT_ATOL, "rtol": LOGIT_RTOL},
               "primary_verdict_changed": False, "profiling_path_changed": False}
    REPORT["hybrid_diagnostic"] = summary
    save_json("compatibility_report.json", REPORT)
    rows, observations, snapshots = [], {}, {}
    model = None
    if hybrid_model is not None:
        torch.cuda.synchronize(DEVICE)
        hybrid_model = None
        gc.collect()
        torch.cuda.empty_cache()
    try:
        for variant in ["independent_awq", "reconstructed_awq"]:
            print("Diagnostic:", variant, flush=True)
            try:
                if variant == "independent_awq":
                    model = load_endpoint("awq")
                else:
                    model = load_endpoint("fp16")
                    for j in range(28):
                        replace_layer(model, "awq", j)
                ownership_report(model)
                snapshots[variant] = diagnostic_runtime_snapshot(model)
                save_json(relative + f"/runtime_{variant}.json", snapshots[variant])
                # Compile and warm both paths separately; these are not timings.
                fixed_trace(model)
                native = [fixed_trace(model) for _ in range(3)]
                with diagnostic_serial_awq_reduction():
                    fixed_trace(model)
                    serial = [fixed_trace(model) for _ in range(2)]
                observations[variant] = {"native": native, "serial": serial}
                for path, outputs in observations[variant].items():
                    for i in range(1, len(outputs)):
                        rows.append(diagnostic_logit_metrics(f"{variant}_{path}_repeat_{i}_vs_0", outputs[i], outputs[0]))
                save_csv(relative + "/logit_comparisons.csv", rows)
            finally:
                model = None
                gc.collect()
                torch.cuda.empty_cache()
        a, b = observations["independent_awq"], observations["reconstructed_awq"]
        if "awq" in REFERENCES:
            rows.append(diagnostic_logit_metrics("reloaded_native_reference_vs_original_step7",
                                                 a["native"][0], REFERENCES["awq"]))
        for path in ["native", "serial"]:
            for i, actual in enumerate(b[path]):
                rows.append(diagnostic_logit_metrics(f"{path}_reconstructed_{i}_vs_independent_0", actual, a[path][0]))
        differences = diagnostic_state_differences(snapshots["independent_awq"], snapshots["reconstructed_awq"])
        save_json(relative + "/runtime_state_differences.json", differences)
        save_csv(relative + "/logit_comparisons.csv", rows)
        serial_cross = [r for r in rows if r["test"].startswith("serial_reconstructed_")]
        native_repeats = [r for r in rows if "_native_repeat_" in r["test"]]
        summary.update({"status": "COMPLETE", "runtime_state_difference_count": len(differences),
                        "native_repeatability_exceeds_original_tolerance": any(not r["passed_original_tolerance"] for r in native_repeats),
                        "serial_endpoints_match_original_tolerance": all(r["passed_original_tolerance"] for r in serial_cross),
                        "interpretation": "Review native repeatability, runtime state differences, and serial endpoint comparisons together; the main failed check remains unresolved."})
        if summary["serial_endpoints_match_original_tolerance"]:
            print("Serial endpoint comparison passed at the original tolerance.")
        else:
            print("Serial endpoint comparison still differs; inspect runtime_state_differences.json.")
        print("Runtime state differences:", len(differences))
        print("Native repeats exceed tolerance:", summary["native_repeatability_exceeds_original_tolerance"])
    except KeyboardInterrupt:
        summary.update(status="INTERRUPTED")
        raise
    except Exception as exc:
        summary.update(status="FAIL", error_type=type(exc).__name__, error=str(exc))
        (OUT / relative / "diagnostic_error.txt").write_text(traceback.format_exc())
        print("Diagnostic error:", str(exc))
    finally:
        save_json(relative + "/summary.json", summary)
        save_json("compatibility_report.json", REPORT)
        print("Diagnostic saved. Run Step 10 again to download the updated ZIP.")

if (REPORT["checks"]["gpu_endpoints"]["status"] == "PASS"
        and REPORT["checks"]["gpu_hybrids"]["status"] == "FAIL"):
    diagnose_awq_hybrid_failure()
else:
    print("Hybrid-failure diagnostic skipped: no eligible GPU hybrid failure.")


Diagnostic: independent_awq
Diagnostic: reconstructed_awq


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Serial endpoint comparison passed at the original tolerance.
Runtime state differences: 0
Native repeats exceed tolerance: True
Diagnostic saved. Run Step 10 again to download the updated ZIP.


## 9c. Complete the remaining checks after confirmed native kernel variation
For the existing Kaggle session, copy and run the entire code cell below.
It exports the updated ZIP itself. Earlier cells do not need to be repeated.

This cell requires the completed diagnostic to show matching runtime state,
matching serial endpoints, and native repeatability beyond the original
tolerance. It preserves the earlier failure and comparison CSV in a history
folder, rebuilds independent references under split-K 1, checks the mixed and
full AWQ configurations, and restores all 28 original FP16 layers exactly.
It then restores the normal split-K 8 kernel and checks a retained-KV
high-to-low-to-high cycle. Tolerances stay unchanged. Performance profiling
continues to use its original kernel settings.

In [20]:
def complete_remaining_compatibility_checks():
    global hybrid_model
    import awq.modules.linear.gemm as gemm
    diagnostic = REPORT.get("hybrid_diagnostic", {})
    require(diagnostic.get("status") == "COMPLETE", "Run Step 9b successfully first")
    require(diagnostic.get("runtime_state_difference_count") == 0
            and diagnostic.get("serial_endpoints_match_original_tolerance") is True
            and diagnostic.get("native_repeatability_exceeds_original_tolerance") is True,
            "The diagnostic does not establish the prerequisites for this completion path")
    require(all(REPORT["checks"][s]["status"] == "PASS"
                for s in ["download", "config_tokenizer", "tensor_audit", "manifests"]),
            "Earlier checkpoint and tensor checks must pass")
    require({"atol": LOGIT_ATOL, "rtol": LOGIT_RTOL} == diagnostic["original_tolerance"],
            "Keep the diagnostic's original tolerances unchanged")
    require(gemm.TRITON_AVAILABLE and gemm.awq_ext is None,
            "Expected the recorded unfused GEMM/Triton execution path")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    history = f"validation_history/{stamp}"
    (OUT / history).mkdir(parents=True, exist_ok=False)
    # Preserve the original outcome, diagnostic metadata, and numerical errors.
    save_json(history + "/report_before_completion.json", copy.deepcopy(REPORT))
    for name in ["gpu_endpoint_comparisons.csv", "gpu_endpoints_error.txt",
                 "gpu_hybrids_error.txt", "gpu_continuity_error.txt", "RESULT.md"]:
        if (OUT / name).exists():
            shutil.copy2(OUT / name, OUT / history / name)
    REPORT.setdefault("completion_history", []).append(history)
    REPORT["compatibility_execution"] = {
        "endpoint_and_hybrid_equivalence_split_k": 1,
        "retained_kv_cycle_split_k": 8,
        "performance_profiling_split_k": 8,
        "prior_results_folder": history,
        "note": "The original native endpoint mismatch is preserved. The active equivalence checks use serial reduction; native KV continuity is checked separately.",
    }
    REPORT["completion_status"] = "RUNNING"
    save_json("compatibility_report.json", REPORT)
    GPU_ROWS.clear()
    REFERENCES.clear()
    original_kernel = gemm.awq_gemm_triton
    interrupted = False
    try:
        if hybrid_model is not None:
            torch.cuda.synchronize(DEVICE)
            hybrid_model = None
            gc.collect()
            torch.cuda.empty_cache()
        # Rebuild references and hybrids under the same serial setting.
        with diagnostic_serial_awq_reduction():
            run_stage("gpu_endpoints", independent_endpoints, dependencies=("manifests",))
            run_stage("gpu_hybrids", hybrid_checks, dependencies=("gpu_endpoints",))
        require(gemm.awq_gemm_triton is original_kernel,
                "The original native AWQ kernel was not restored")
        # Continuity does not compare AWQ logits between separate runs. Run it
        # on the native path, checking exact cached prefixes and restored state.
        run_stage("gpu_continuity", continuity_check, dependencies=("gpu_hybrids",))
    except KeyboardInterrupt:
        interrupted = True
        REPORT["completion_status"] = "INTERRUPTED"
        raise
    except Exception as exc:
        record("gpu_hybrids", "FAIL", error_type=type(exc).__name__, error=str(exc))
        record("gpu_continuity", "NOT_RUN", reason="Completion runner failed")
        (OUT / "completion_error.txt").write_text(traceback.format_exc())
    finally:
        # The context manager restores the kernel even after a failed check.
        if hybrid_model is not None:
            hybrid_model = None
        gc.collect()
        torch.cuda.empty_cache()
        for stage, split_k in [("gpu_endpoints", 1), ("gpu_hybrids", 1), ("gpu_continuity", 8)]:
            check = REPORT["checks"][stage]
            check["awq_split_k_for_check"] = split_k
            if check["status"] == "PASS":
                (OUT / f"{stage}_error.txt").unlink(missing_ok=True)
        statuses = {k: v["status"] for k, v in REPORT["checks"].items()}
        verdict = ("FAIL" if "FAIL" in statuses.values() else
                   "PASS" if all(s == "PASS" for s in statuses.values()) else "INCOMPLETE")
        REPORT["verdict"] = verdict
        if not interrupted:
            REPORT["completion_status"] = verdict
        REPORT["next_step"] = ("Review serial equivalence, exact FP16 restoration, and native-path KV continuity; then proceed to switching-cost profiling."
                               if verdict == "PASS" else "Resolve the remaining failed or incomplete checks.")
        save_json("compatibility_report.json", REPORT)

    scope = ("Endpoint and hybrid equivalence: split-K 1. Retained-KV cycle: native split-K 8. "
             "Performance profiling retains split-K 8. Original numerical mismatch preserved in validation_history.")
    lines = ["# ReServe compatibility result", "", f"**Verdict: {verdict}**", "", scope, "",
             "| Stage | Status |", "| --- | --- |"]
    lines += [f"| {k} | {v} |" for k, v in statuses.items()]
    lines += ["", REPORT["next_step"], "",
              "This compatibility result does not validate switching costs or an elastic page allocator."]
    (OUT / "RESULT.md").write_text("\n".join(lines) + "\n")
    (OUT / "CORRECTNESS_EXECUTION.md").write_text(scope + "\n")
    archive = Path(shutil.make_archive(str(OUT), "zip", root_dir=str(OUT)))
    print("COMPATIBILITY VERDICT:", verdict)
    print(json.dumps(statuses, indent=2))
    print(scope)
    from IPython.display import FileLink, Markdown, display
    display(Markdown("\n".join(lines)))
    display(FileLink(str(archive)))

diagnostic = REPORT.get("hybrid_diagnostic", {})
if (diagnostic.get("status") == "COMPLETE"
        and diagnostic.get("runtime_state_difference_count") == 0
        and diagnostic.get("serial_endpoints_match_original_tolerance") is True
        and diagnostic.get("native_repeatability_exceeds_original_tolerance") is True):
    complete_remaining_compatibility_checks()
else:
    print("Step 9c skipped: a completed diagnostic confirming the native-kernel variation case is required.")


gpu_endpoints: RUNNING
fp16 reference: finite prefill, decode, and FP16 KV


/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['version', 'fuse_max_seq_len', 'exllama_config', 'modules_to_fuse', 'do_fuse']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)


awq reference: finite prefill, decode, and FP16 KV
gpu_endpoints: PASS
gpu_hybrids: RUNNING
fixture_layer_0 passed
fixture_layers_0_13_27 passed
all_layers_awq passed
gpu_hybrids: PASS
gpu_continuity: RUNNING
gpu_continuity: PASS
COMPATIBILITY VERDICT: PASS
{
  "download": "PASS",
  "config_tokenizer": "PASS",
  "tensor_audit": "PASS",
  "manifests": "PASS",
  "gpu_endpoints": "PASS",
  "gpu_hybrids": "PASS",
  "gpu_continuity": "PASS"
}
Endpoint and hybrid equivalence: split-K 1. Retained-KV cycle: native split-K 8. Performance profiling retains split-K 8. Original numerical mismatch preserved in validation_history.


# ReServe compatibility result

**Verdict: PASS**

Endpoint and hybrid equivalence: split-K 1. Retained-KV cycle: native split-K 8. Performance profiling retains split-K 8. Original numerical mismatch preserved in validation_history.

| Stage | Status |
| --- | --- |
| download | PASS |
| config_tokenizer | PASS |
| tensor_audit | PASS |
| manifests | PASS |
| gpu_endpoints | PASS |
| gpu_hybrids | PASS |
| gpu_continuity | PASS |

Review serial equivalence, exact FP16 restoration, and native-path KV continuity; then proceed to switching-cost profiling.

This compatibility result does not validate switching costs or an elastic page allocator.

/kaggle/working/compatibility_fp16_awq_20261127T191507_288845Z.zip

## 10. Export the verdict and download the ZIP

Run this cell even if another check fails. Download the ZIP and provide it
for review before writing the switching-cost profiler.

| Verdict | Meaning |
| --- | --- |
| PASS | Static checks, endpoint equivalence, representative hybrids, exact restoration, and the small retained-KV cycle passed. |
| FAIL | At least one check found a mismatch or execution error. Inspect the named stage and saved error; do not silently combine checkpoints. |
| INCOMPLETE | A required check was skipped, interrupted, or not reached. Static results alone do not establish executable layer compatibility. |

A PASS applies to the tested pinned variants and this reference boundary.
Switching time, transient allocation calibration, final LIS order, output
quality, and elastic KV page detachment remain separate work.


In [21]:
statuses = {k: v["status"] for k, v in REPORT["checks"].items()}
if any(s == "FAIL" for s in statuses.values()):
    verdict = "FAIL"
elif all(s == "PASS" for s in statuses.values()):
    verdict = "PASS"
else:
    verdict = "INCOMPLETE"
REPORT["verdict"] = verdict
REPORT["next_step"] = ("Review the compatibility report, then implement directional switching-cost profiling."
                        if verdict == "PASS" else "Resolve or complete the listed checks before treating layer variants as interchangeable.")
save_json("compatibility_report.json", REPORT)
lines = ["# ReServe compatibility result", "", f"**Verdict: {verdict}**", "",
         "| Stage | Status |", "| --- | --- |"]
lines += [f"| {k} | {v} |" for k, v in statuses.items()]
lines += ["", REPORT["next_step"], "",
          "This report does not establish switching-cost or elastic allocator measurements."]
(OUT / "RESULT.md").write_text("\n".join(lines) + "\n")
# Archive report data only, without weights, HF tokens, or the model cache.
archive = Path(shutil.make_archive(str(OUT), "zip", root_dir=str(OUT)))
print("VERDICT:", verdict)
print(json.dumps(statuses, indent=2))
print("ZIP:", archive)
from IPython.display import FileLink, Markdown, display
display(Markdown("\n".join(lines)))
display(FileLink(str(archive)))


VERDICT: PASS
{
  "download": "PASS",
  "config_tokenizer": "PASS",
  "tensor_audit": "PASS",
  "manifests": "PASS",
  "gpu_endpoints": "PASS",
  "gpu_hybrids": "PASS",
  "gpu_continuity": "PASS"
}
ZIP: /kaggle/working/compatibility_fp16_awq_20261127T191507_288845Z.zip


# ReServe compatibility result

**Verdict: PASS**

| Stage | Status |
| --- | --- |
| download | PASS |
| config_tokenizer | PASS |
| tensor_audit | PASS |
| manifests | PASS |
| gpu_endpoints | PASS |
| gpu_hybrids | PASS |
| gpu_continuity | PASS |

Review the compatibility report, then implement directional switching-cost profiling.

This report does not establish switching-cost or elastic allocator measurements.

/kaggle/working/compatibility_fp16_awq_20261127T191507_288845Z.zip

## API references

- [Pinned original checkpoint](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/tree/989aa7980e4cf806f80c7fef2b1adb7bc71aa306)
- [Pinned AWQ checkpoint](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-AWQ/tree/c5b72040769bfde6c5b562ae890f9657b0803f79)
- [Transformers 4.51.3 Qwen2 implementation](https://github.com/huggingface/transformers/blob/v4.51.3/src/transformers/models/qwen2/modeling_qwen2.py)
- [Transformers 4.51.3 AWQ integration](https://github.com/huggingface/transformers/blob/v4.51.3/src/transformers/integrations/awq.py)
- [AutoAWQ GEMM implementation](https://github.com/casper-hansen/AutoAWQ/blob/main/awq/modules/linear/gemm.py)
- [Safetensors usage](https://huggingface.co/docs/safetensors/en/index)
